In [1]:
import pandas as pd
import os
import warnings
import joblib

os.chdir('/home/LULAB/wboohar/CODEX/data_processing/code')
from quantcell import quantcell_project
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier

In [2]:
annotations_path = '/store/Projects/wboohar/PhenoCycler/annotation_strategies/marker_combos_062525_updated_verified.json'   
base_dir = '/store/Projects/wboohar/PhenoCycler' 
project_name = 'QuantCellPaper'
project_path = f'{base_dir}/{project_name}'
data_path = f'{base_dir}/raw_data'

project_folders = ['122123BW Annotations']
# sample ages in file are correct

sample_labels = {'122123BW Annotations':'QuantCellPaper'}


project_folders_replace = ['012925BW Y1 Annotations', '012925BW Y3 Annoations', '013025BW O1 Annotations', '013125BW O2 Annotations', '013125BW O3 Annotation', '020425BW Y2 Annotations']
sample_labels_replace = {
    '012925BW Y1 Annotations' : 'Y1',
    '012925BW Y3 Annoations' : 'Y3',
    '013025BW O1 Annotations' : 'O1',
    '013125BW O2 Annotations' : 'O2',
    '013125BW O3 Annotation' : 'O3',
    '020425BW Y2 Annotations' : 'Y2'
}

In [3]:
project = quantcell_project()
project.initialize(base_path=base_dir, project_name=project_name)

Initializing project QuantCellPaper at /store/Projects/wboohar/PhenoCycler


In [4]:
project.process_data()

In [5]:
best_params = joblib.load(f'{project_path}/model_selection/RandomForestClassifier_best_params_macro.joblib')

classifier = RandomForestClassifier(
    **best_params,
    n_jobs=36,
    random_state=project.random_seed
)

project.set_clf(classifier)

In [6]:
project.fit()

In [7]:
project.relabel_unannotated(FDR_cutoff=0.05)

In [8]:
project.overwrite_codex()
project.save_codex()

In [ ]:
from scipy.spatial import Voronoi, voronoi_plot_2d, Delaunay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(dpi=500, figsize=(10, 10))
section = project.codex.loc[project.codex.section == '122123BW Annotations_0', :]
section.reset_index(inplace=True, drop=True)
vor = Voronoi(section.loc[:, ['x', 'y']])
fig = voronoi_plot_2d(vor, show_vertices=False, show_points=False, line_width=0.05, ax=ax)
axs = fig.axes
plt.xlim([4385, 4520])
plt.ylim([21775, 21950])

colors = []
#for i in section.index:
  #  if '+' in section.loc[i, 'CD117']:
  #      colors.append('green')
  #  elif section.loc[i, 'CD117: Cell: Max'] < 10:
  #      colors.append('red')
  #  else:
  #      colors.append('white')


for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color = c_dict[section.loc[r, 'cell_type']], edgecolor='none')

In [ ]:
from scipy.spatial import Voronoi, voronoi_plot_2d, Delaunay

c_dict = {"Arteries" : 'rosybrown',
          "B Cells" : 'violet',
          "Capillaries" : 'brown',
          "CD4 T Cells" : 'red',
          "CD8 T Cells" : 'orangered',
          'cDC' : 'chocolate',
          'CLP' : 'darkorange',
          'CMP' : 'orange',
          'CF4-f' : 'sandybrown',
          "Endothelial Niche" : 'orange',
          "Erythrocytes" : 'wheat',
          "Erythroblasts" : 'gold',
          'GMP' : 'olive',
          "KLS" : 'yellowgreen',
          "Monocytes/Macrophages": 'chartreuse',
          'Megakaryocytes': 'darkseagreen',
          'MEP' : 'palegreen',
          "Mesenchymal Stromal Cells" : 'limegreen',
          "CFU-E": 'green',
          "Myeloid Progenitor Cells" :  'aquamarine',
          'Neutrophils' : 'teal',
          "Perivascular Niche" : 'deepskyblue',
          "Platelets": 'cornflowerblue',
          'Double Negative T Cells' : 'blue',
          "Sinusoids" : 'deeppink',
          "Other" : 'white'}
fig, ax = plt.subplots(dpi=500, figsize=(10, 10))
section = relabeled_codex.loc[project.codex.section == '102424BW M1_1B Annotations_1', :]
section.reset_index(inplace=True, drop=True)
vor = Voronoi(section.loc[:, ['x', 'y']])
fig = voronoi_plot_2d(vor, show_vertices=False, show_points=False, line_width=0.05, ax=ax)
axs = fig.axes
plt.xlim([4385, 4520])
plt.ylim([21775, 21950])

colors = []
#for i in section.index:
  #  if '+' in section.loc[i, 'CD117']:
  #      colors.append('green')
  #  elif section.loc[i, 'CD117: Cell: Max'] < 10:
  #      colors.append('red')
  #  else:
  #      colors.append('white')


for r in range(len(vor.point_region)):
    region = vor.regions[vor.point_region[r]]
    if not -1 in region:
        polygon = [vor.vertices[i] for i in region]
        plt.fill(*zip(*polygon), color = c_dict[section.loc[r, 'cell_type']], edgecolor='none')

In [ ]:
section.cell_type.value_counts()/len(section)